# Transformer：注意力就是你所需要的
> 难度标签：高级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：06_深度学习 | 源文件：Transformer.py | 核心功能：自注意力机制、多头注意力、位置编码、编码器-解码器架构

## 概述

Transformer 是 2017 年 Google 论文 "Attention Is All You Need" 提出的架构，彻底改变了 NLP 领域，现在已扩展到 CV、音频等多个领域。核心创新：用**自注意力机制**替代 RNN 的循环结构，实现完全并行化。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import math

## 数学原理

### 1. 自注意力机制（Scaled Dot-Product Attention）

**代码对应**：Transformer 的核心是自注意力。

输入序列 $\mathbf{X} \in \mathbb{R}^{n \times d}$，通过线性变换得到 Query、Key、Value：

$$\mathbf{Q} = \mathbf{X}\mathbf{W}_Q, \quad \mathbf{K} = \mathbf{X}\mathbf{W}_K, \quad \mathbf{V} = \mathbf{X}\mathbf{W}_V$$

注意力计算：

$$\text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\left(\frac{\mathbf{Q}\mathbf{K}^T}{\sqrt{d_k}}\right)\mathbf{V}$$

除以 $\sqrt{d_k}$ 防止点积过大导致 softmax 梯度消失。

### 2. 多头注意力

**代码对应**：`nn.MultiheadAttention(embed_dim, num_heads)`。

将 Q、K、V 拆分为 $h$ 个头，每个头独立计算注意力，然后拼接：

$$\text{MultiHead}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)\mathbf{W}_O$$

$$\text{head}_i = \text{Attention}(\mathbf{Q}\mathbf{W}_Q^i, \mathbf{K}\mathbf{W}_K^i, \mathbf{V}\mathbf{W}_V^i)$$

多头让模型同时关注不同位置的不同表示子空间。

### 3. 位置编码

Transformer 没有递归结构，需要位置编码注入序列顺序信息：

$$PE_{(pos, 2i)} = \sin(pos / 10000^{2i/d}), \quad PE_{(pos, 2i+1)} = \cos(pos / 10000^{2i/d})$$

正弦位置编码的优势：$PE_{pos+k}$ 可以表示为 $PE_{pos}$ 的线性函数，使模型能学习相对位置关系。

### 4. Feed-Forward Network

每个 Transformer 层包含一个位置-wise 的 FFN：

$$\text{FFN}(\mathbf{x}) = \text{ReLU}(\mathbf{x}\mathbf{W}_1 + \mathbf{b}_1)\mathbf{W}_2 + \mathbf{b}_2$$

### 5. Layer Normalization

$$\text{LN}(\mathbf{x}) = \gamma \odot \frac{\mathbf{x} - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

其中 $\mu, \sigma^2$ 在特征维度上计算（不是 batch 维度）。

### 6. 计算复杂度

自注意力的计算复杂度为 $O(n^2 d)$（$n$ 为序列长度），内存复杂度 $O(n^2)$。这是 Transformer 处理长序列的主要瓶颈。

### 1. 位置编码

运行 1. 位置编码 的代码，观察算法在该环节的行为。

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
# --- 继续 ---
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer("pe", pe)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        return x + self.pe[:, :x.size(1)]

### 2. 简化 Transformer 编码器

运行 2. 简化 Transformer 编码器 的代码，观察算法在该环节的行为。

In [ ]:
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size=1000, d_model=64, nhead=4, num_layers=2,
                 dim_ff=128, n_classes=10, max_len=200, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
# --- 继续 ---
        self.pos_enc = PositionalEncoding(d_model, max_len)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True
        )
# --- 继续 ---
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(d_model, n_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: (batch, seq_len) - token indices
        x = self.embedding(x) * math.sqrt(self.embedding.embedding_dim)
        x = self.pos_enc(x)

        # 生成注意力掩码（padding mask）
        src_key_padding_mask = (x.sum(dim=-1) == 0)  # 简化：全零为 padding

        x = self.transformer(x, src_key_padding_mask=src_key_padding_mask)
        # 取 [CLS] 或平均池化
        x = x.mean(dim=1)  # 平均池化
        x = self.dropout(x)
        x = self.fc(x)
        return x

### 3. 自注意力机制详解

运行 3. 自注意力机制详解 的代码，观察算法在该环节的行为。

In [ ]:
print("=== 自注意力机制 (Self-Attention) ===")
print("Q = X × W_Q, K = X × W_K, V = X × W_V")
print("Attention(Q,K,V) = softmax(Q × K^T / √d_k) × V")
print()
print("多头注意力 (Multi-Head Attention):")
# --- 输出结果 ---
print("  将 Q,K,V 分成 h 个头，分别计算注意力后拼接")
print("  MultiHead = Concat(head_1, ..., head_h) × W_O")
print("  每个头关注不同的子空间信息")

### 4. 构造分类数据

在分类任务上训练模型并评估性能。

In [ ]:
np.random.seed(42)
torch.manual_seed(42)

# 模拟文本分类数据
n_samples = 1000
vocab_size = 1000
max_len = 50
n_classes = 5

# 生成随机序列和标签
X = torch.randint(1, vocab_size, (n_samples, max_len))
# 简单规则：前几个 token 的均值决定类别
y = (X[:, :5].float().mean(dim=1) % n_classes).long()

X_train, X_test = X[:800], X[800:]
y_train, y_test = y[:800], y[800:]

### 5. 训练 Transformer

执行模型训练循环，观察损失函数的收敛过程。

In [ ]:
model = TransformerClassifier(vocab_size=vocab_size, n_classes=n_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

print(f"\n=== Transformer 模型结构 ===")
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"总参数量: {total_params:,}")

print("\n=== 训练 ===")
n_epochs = 30
for epoch in range(n_epochs):
    model.train()
    output = model(X_train)
# --- 继续 ---
    loss = criterion(output, y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        model.eval()
        with torch.no_grad():
            train_acc = (model(X_train).argmax(1) == y_train).float().mean().item()
            test_acc = (model(X_test).argmax(1) == y_test).float().mean().item()
# --- 输出结果 ---
        print(f"  Epoch {epoch+1:>2}: Loss={loss.item():.4f}, "
              f"Train Acc={train_acc:.4f}, Test Acc={test_acc:.4f}")

### 6. 注意力权重可视化

运行 6. 注意力权重可视化 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 注意力权重分析 ===")
# 获取注意力权重需要修改模型输出中间层，此处简化演示
with torch.no_grad():
    sample = X_test[:1]
    emb = model.embedding(sample) * math.sqrt(model.embedding.embedding_dim)
    emb = model.pos_enc(emb)
    # 手动计算 Q, K
    q = model.transformer.layers[0].self_attn.in_proj_weight[:64]  # 简化
    print(f"嵌入维度: {emb.shape}")
    print(f"输出维度: {model.transformer(emb).shape}")

print("\n=== Transformer vs RNN/LSTM ===")
print("Transformer: 并行计算所有位置（快），捕捉全局依赖（好）")
print("RNN/LSTM: 序列计算（慢），长距离依赖受限")
print("但 Transformer 对长序列的内存消耗大（O(n²)）")

print("\n=== Transformer 要点 ===")
print("- 自注意力: O(1) 路径长度，直接建模任意距离的依赖")
print("- 多头注意力: 每个头学习不同的关注模式")
print("- 位置编码: 注入序列顺序信息（Transformer 本身无位置感知）")
print("- 残差连接 + LayerNorm: 稳定训练")
# --- 输出结果 ---
print("- 应用: BERT(编码), GPT(解码), Vision Transformer(图像)")

## 关键代码解释

### 自注意力

```python
Attention(Q, K, V) = softmax(Q @ K^T / sqrt(d_k)) @ V
```

Q（查询）、K（键）、V（值）是输入的线性变换。注意力分数衡量"每个位置应该关注其他位置多少"。

### 多头注意力

```python
nn.MultiheadAttention(embed_dim=512, num_heads=8)
```

多个注意力头并行计算，每个头关注不同的子空间信息。

### 位置编码

Transformer 没有循环结构，需要位置编码告诉模型每个 token 的位置。通常用正弦/余弦函数或可学习的位置嵌入。

## 注意事项

1. **O(n^2) 复杂度**：序列长度的平方，长序列很慢
2. **位置编码很重要**：没有位置信息，模型无法区分序列顺序
3. **需要大量数据**：参数多，小数据集容易过拟合

## 总结与延伸

以上代码展示了 **Transformer** 的完整流程。

**解读要点**：
- 关注 **损失函数** 的收敛曲线
- 观察训练/验证损失是否出现分叉（过拟合信号）
- 对比不同超参数（学习率、层数）的影响

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **Flash Attention**：IO 感知的高效注意力实现
- **Vision Transformer (ViT)**：Transformer 在图像领域的应用
- **线性注意力**：O(n) 复杂度的近似注意力
